# Assignment 4: Frontier ML: Advanced Learning Paradigms

**Student Name:** Sebastian Campos      

**Student ID:** [redacted]

**Points:** 11

## Assignment Overview

This assignment tours the advanced learning paradigms of the second half of the course. Problem 1 builds a recommender and a multilabel classifier on one anime ratings dataset. Problem 2 covers transfer learning and semi-supervised learning. Problem 3 mines structure from a network. Problem 4 simulates federated learning.

**Note:** Use your Student ID as the random seed everywhere a seed is required. Report results over **three seeds** wherever a problem says so.

**Compute note.** If your computer runs out of memory or time, reduce the sample size (fewer users or anime in Problem 1, fewer or smaller images and fewer epochs in Problem 2a, a smaller Covertype subsample in Problem 4) and still complete every part of the assignment. State exactly what you reduced, and by how much, in a short note at the top of your notebook. A smaller run that you finish and explain earns full credit for method; a run that does not finish earns none. 

## Dataset Information

| Problem | Dataset | Exact source |
|---|---|---|
| 1a, 1b | **Anime Recommendations Database** (MyAnimeList data, CooperUnion): `anime.csv` (`anime_id, name, genre, type, episodes, rating, members`) and `rating.csv` (`user_id, anime_id, rating`; -1 means watched but not scored). About 7.8 million ratings, 73,000 users, 12,000 anime. | Kaggle page: https://www.kaggle.com/datasets/CooperUnion/anime-recommendations-database (free Kaggle account; click Download, unzip to get both CSV files). A copy is also on Canvas. |
| 2a | **CIFAR-10** (Krizhevsky, 2009) | Official page: https://www.cs.toronto.edu/~kriz/cifar.html ; in code, `torchvision.datasets.CIFAR10(root="data", download=True)` |
| 2b | Breast Cancer Wisconsin (Diagnostic) | Built in: `sklearn.datasets.load_breast_cancer()` ; original source https://archive.ics.uci.edu/dataset/17 |
| 3 | Zachary's Karate Club graph | Built in: `networkx.karate_club_graph()` ; docs https://networkx.org/documentation/stable/reference/generated/networkx.generators.social.karate_club_graph.html |
| 4 | **Forest Covertype** (581,012 cells, 7 forest types, 54 features) | In code: `sklearn.datasets.fetch_covtype()` ; original source https://archive.ics.uci.edu/dataset/31/covertype |

**Loading the anime files.** Put `anime.csv` and `rating.csv` in your working folder and read them with `pd.read_csv`. Do not commit the 7.8-million-row file to GitHub; work from the subset below.

**Working subset for Problem 1 (state it in your notebook).** Keep the 2,000 most-rated anime and a random sample of 10,000 users who rated at least 20 of them; drop ratings of -1. Report the final number of users, items, and ratings, and the density.

## Import Required Libraries

Import all necessary libraries for data analysis, visualization, and machine learning.

In [ ]:
# Import necessary libraries for data analysis, visualization, and machine learning
# YOUR CODE HERE:
import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, accuracy_score, hamming_loss,classification_report,roc_auc_score,  normalized_mutual_info_score
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import ClassifierChain
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from scipy.sparse import hstack
import html
from sklearn.dummy import DummyClassifier
import torchvision
import torchvision.transforms as T
import warnings
from torch.utils.data import Subset
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.semi_supervised import SelfTrainingClassifier
import networkx as nx
from sklearn.datasets import fetch_covtype
import os

---
## Problem 1: Recommender Systems & Multilabel Classification  (3 points)

### Problem 1a: Recommender Evaluated as a Ranking Task (1.5 points)

**Instructions:**

Build recommenders on the Problem 1 subset and judge them by how well they rank, not only by rating error.

* Split: hold out a random 20% of each user's ratings (seeded). A held-out rating of 8 or higher counts as *relevant*.
* Required models: (i) a **popularity** baseline, (ii) an **item-based collaborative filter** (cosine similarity), and (iii) **one latent-factor model** of your choice (implicit ALS, truncated SVD, or similar).
* Metrics: precision@10, recall@10 and NDCG@10 over all users who have at least one relevant held-out item (the same protocol as the class lab; a rating of 8 on the 10-point scale plays the role of the lab's 4 on a 5-point scale). Exclude items the user already rated in training. Show a random baseline and the three models in one table.
* Also print the top-5 recommendations (anime names) for one user of your choice.

In [ ]:
# TODO: Load rating.csv and anime.csv, build the working subset, report users/items/ratings/density
# TODO: Split 80/20 per user; define relevant = held-out rating >= 8
# TODO: Write evaluate(score_matrix) returning precision@10, recall@10 and NDCG@10
# TODO: Build popularity, item-based CF, and one latent-factor model; evaluate all three
# TODO: Print the top-5 recommended anime for one chosen user
# YOUR CODE HERE:

# Load the anime and ratings datasets
anime = pd.read_csv("animedatabase/anime.csv")
ratings = pd.read_csv("animedatabase/rating.csv")

# Filter out invalid ratings (where rating is -1)
valid_ratings = ratings[ratings['rating'] != -1]

anime_counts = valid_ratings['anime_id'].value_counts()
top_anime_ids = anime_counts.head(2000).index
sub = valid_ratings[valid_ratings['anime_id'].isin(top_anime_ids)]


user_count = sub['user_id'].value_counts()
eligible_users = user_count[user_count >= 20].index

userseed = int(os.environ["STUDENT_ID"])
randomNumberGenerator = np.random.default_rng(seed=userseed)
sampled_users = randomNumberGenerator.choice(eligible_users, size=10000, replace=False)

n_user = sub[sub['user_id'].isin(sampled_users)]['user_id'].nunique()
n_item = sub[sub['user_id'].isin(sampled_users)]['anime_id'].nunique()
n_rating = sub[sub['user_id'].isin(sampled_users)].shape[0]         
density = n_rating / (n_user * n_item)

print(f"Users: {n_user:,}  Items: {n_item:,}  Ratings: {n_rating:,}  Density: {density:.4f}")

# Finalize the working subset from your sampled users
working_subset = sub[sub['user_id'].isin(sampled_users)].copy()

# Shuffle using your seed to randomize each user's interaction order
shuffled_subset = working_subset.sample(frac=1.0, random_state=userseed).reset_index(drop=True)

# Assign a normalized position (0.0 to 1.0) within each user's interactions
shuffled_subset['user_rank'] = shuffled_subset.groupby('user_id').cumcount()
shuffled_subset['user_total'] = shuffled_subset.groupby('user_id')['anime_id'].transform('count')
shuffled_subset['split_ratio'] = shuffled_subset['user_rank'] / shuffled_subset['user_total']

# Split 80% into train and 20% into test per user
train_df = shuffled_subset[shuffled_subset['split_ratio'] < 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])
test_df = shuffled_subset[shuffled_subset['split_ratio'] >= 0.8].drop(columns=['user_rank', 'user_total', 'split_ratio'])


# Define relevant items in the held-out set (rating >= 8)
test_df['relevant'] = (test_df['rating'] >= 8).astype(int)

# Map each user_id to their set of relevant held-out anime_ids
ground_truth_relevant = (
    test_df[test_df['relevant'] == 1]
    .groupby('user_id')['anime_id']
    .apply(set)
    .to_dict()
)

# ID -> matrix index mappings (fixed once, shared by every model)
user_ids  = np.sort(working_subset['user_id'].unique())
anime_ids = np.sort(working_subset['anime_id'].unique())
user_to_row = {u: i for i, u in enumerate(user_ids)}
anime_to_col = {a: j for j, a in enumerate(anime_ids)}

# Items each user rated in training (column indices)
train_items_real = (
    train_df.assign(row=train_df['user_id'].map(user_to_row),
                    col=train_df['anime_id'].map(anime_to_col))
    .groupby('row')['col'].apply(set).to_dict()
)

# Relevant held-out items (column indices)
relevant_real = {
    user_to_row[u]: {anime_to_col[a] for a in items}
    for u, items in ground_truth_relevant.items()
}


def evaluate(score_matrix, K=10, train_items=train_items_real, relevant=relevant_real):
    precisions, recalls, ndcg_scores = [], [], []

    for u, rel in relevant.items():
        row = score_matrix[u].copy()
        row[list(train_items.get(u, set()))] = -np.inf   # exclude training items

        top_k = np.argsort(-row)[:K]
        hits = [1 if item in rel else 0 for item in top_k]

        precisions.append(sum(hits) / K)
        recalls.append(sum(hits) / len(rel))

        dcg  = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1 / np.log2(i + 2) for i in range(min(len(rel), K)))
        ndcg_scores.append(dcg / idcg)

    return np.mean(precisions), np.mean(recalls), np.mean(ndcg_scores)


# Sanity check 1: toy example should match your earlier numbers
toy_scores = np.array([
    [0.9, 0.1, 0.8, 0.3, 0.7, 0.2, 0.6, 0.4],
    [0.2, 0.9, 0.1, 0.8, 0.3, 0.7, 0.4, 0.6],
    [0.5, 0.4, 0.3, 0.2, 0.9, 0.8, 0.1, 0.7],
])
toy_train = {0: {0}, 1: {1, 3}, 2: set()}
toy_rel   = {0: {2, 6}, 1: {5}, 2: {4, 7, 1}}
print("Toy:", evaluate(toy_scores, K=3, train_items=toy_train, relevant=toy_rel))

# Sanity check 2: random scores on real data should be near zero
rand_scores = np.random.default_rng(userseed).random((len(user_ids), len(anime_ids)))
p, r, n = evaluate(rand_scores)
print(f"Random  P@10: {p:.4f}  R@10: {r:.4f}  NDCG@10: {n:.4f}")

# 1. Build the training user-item matrix
n_users = len(user_ids)
n_items = len(anime_ids)

train_rows = train_df['user_id'].map(user_to_row).values
train_cols = train_df['anime_id'].map(anime_to_col).values
train_vals = train_df['rating'].values.astype(np.float32)

R_train = csr_matrix((train_vals, (train_rows, train_cols)), shape=(n_users, n_items))

# Model 1: Popularity baseline (number of training ratings per anime)
item_counts = np.asarray((R_train > 0).sum(axis=0)).ravel().astype(np.float32)
pop_scores = np.tile(item_counts, (n_users, 1))      # same ranking for every user

# Model 2: Item-based collaborative filtering (cosine similarity)
item_sim = cosine_similarity(R_train.T)              # 2000 x 2000 item-item similarity
np.fill_diagonal(item_sim, 0.0)                      # an item should not recommend itself
item_cf_scores = R_train @ item_sim                  # user's ratings weighted by similarity

# Model 3: Latent-factor model (truncated SVD)
svd = TruncatedSVD(n_components=32, random_state=userseed)
U_latent = svd.fit_transform(R_train)                # user factors (include singular values)
V_latent = svd.components_                           # item factors
latent_scores = U_latent @ V_latent

# Evaluate all models
models = {
    "Random": rand_scores,
    "Popularity": pop_scores,
    "Item-Based CF": item_cf_scores,
    "Latent Factor (SVD)": latent_scores,
}

results = []
for name, scores in models.items():
    p, r, ndcg = evaluate(scores, K=10)
    results.append({"Model": name, "P@10": p, "R@10": r, "NDCG@10": ndcg})

print("\n" + pd.DataFrame(results).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

id_to_name = dict(zip(anime['anime_id'], anime['name']))

sample_u = next(iter(relevant_real.keys()))
sample_user_id = user_ids[sample_u]


user_train_df =(
    train_df[train_df['user_id']==sample_user_id]
    .sort_values('rating', ascending= False)
    .head(5)
)

# Top-5 recommendations from the best model (SVD)
row = latent_scores[sample_u].copy()
row[list(train_items_real.get(sample_u, set()))] = -np.inf   # skip anime already rated in training

top5_cols = np.argsort(-row)[:5]
top5_ids = anime_ids[top5_cols]

print(f"User {sample_user_id}: top-rated anime in training")
for _, r in user_train_df.iterrows():
    print(f"  {id_to_name.get(r['anime_id'], r['anime_id'])}  (rated {r['rating']})")

print(f"\nTop-5 SVD recommendations for user {sample_user_id}")
for rank, a in enumerate(top5_ids, 1):
    print(f"  {rank}. {id_to_name.get(a, a)}")

### 📝 Analysis Explanation (1a)

**Instructions:** Which model ranks best, and does the ordering match what you would expect from the class lab? Explain why the popularity baseline can be hard to beat. What is the cold-start problem, and how would it affect your best model for a brand-new anime or a brand-new user?

**Your Analysis:** [Write your interpretation here.]

### Problem 1b: Multilabel Genre Classification (1.5 points)

**Instructions:**

Each anime lists several genres in the `genre` column (comma-separated), so genre prediction is a multilabel problem. Keep the 12 most frequent genres as labels and the anime that have at least one of them.

* Features, built per anime: `type` (one-hot), `episodes`, `members`, a TF-IDF of the `name`, and **audience features** (a 30-component truncated SVD of the anime-by-user matrix (as in the class lab) from your training ratings).
* Models: a trivial baseline (always predict the most common genre), **binary relevance** (one logistic regression per genre), and a **classifier chain**.
* Evaluate with Hamming loss, subset accuracy, micro F1, and macro F1 on a held-out 25% of anime. Compare feature sets: name text only, audience only, all features.

In [ ]:
# TODO: Build the 12-genre label matrix and the three feature sets (text, audience SVD, all)
# TODO: Train the always-most-common baseline, binary relevance, and a classifier chain
# TODO: Report Hamming loss, subset accuracy, micro F1, macro F1 for each model and feature set in one table
# TODO: Print per-genre F1 for your best model
# YOUR CODE HERE:
# Labels for the 2,000 anime in the ratings matrix, in the same order as anime_ids / R_train columns
anime_sub = anime.set_index('anime_id').reindex(anime_ids)

genre_lists = (
    anime_sub['genre']
    .fillna('')
    .apply(lambda x: [g.strip() for g in x.split(',') if g.strip()])
)

top_12_genres = (
    genre_lists.explode()
    .value_counts()
    .head(12)
    .index
    .tolist()
)

filtered_genres = genre_lists.apply(lambda gs: [g for g in gs if g in top_12_genres])

mlb = MultiLabelBinarizer(classes=top_12_genres)
Y_genres = mlb.fit_transform(filtered_genres)

# Keep only anime with at least one of the 12 genres
keep_mask = Y_genres.sum(axis=1) > 0
Y_genres = Y_genres[keep_mask]

print("Label matrix shape:", Y_genres.shape)
print("Genres:", mlb.classes_)

# ---------- Features (one row per anime, same order as anime_sub / anime_ids) ----------
names = anime_sub['name'].fillna('').apply(html.unescape)

# type one-hot
type_onehot = pd.get_dummies(anime_sub['type'].fillna('Unknown'), prefix='type').astype(float).values

# episodes ("Unknown" -> NaN -> median) and members, log-scaled because both are very skewed
episodes = pd.to_numeric(anime_sub['episodes'], errors='coerce')
episodes = np.log1p(episodes.fillna(episodes.median())).values.reshape(-1, 1)
members = np.log1p(anime_sub['members'].fillna(0)).values.reshape(-1, 1)

# Audience features: 30-component SVD of the anime-by-user matrix from the TRAINING ratings
audience_svd = TruncatedSVD(n_components=30, random_state=userseed).fit_transform(R_train.T)

# Keep only anime with at least one of the 12 genres (same rows as Y_genres)
names        = names[keep_mask].reset_index(drop=True)
type_onehot  = type_onehot[keep_mask]
episodes     = episodes[keep_mask]
members      = members[keep_mask]
audience_svd = audience_svd[keep_mask]

# ---------- Train/test split: hold out 25% of anime ----------
idx_train, idx_test = train_test_split(np.arange(len(Y_genres)), test_size=0.25, random_state=userseed)
Y_train, Y_test = Y_genres[idx_train], Y_genres[idx_test]

# TF-IDF of the name (fit on training anime only)
tfidf = TfidfVectorizer(min_df=2)
text_train = tfidf.fit_transform(names.iloc[idx_train])
text_test  = tfidf.transform(names.iloc[idx_test])

# Audience SVD, scaled (fit on training anime only)
aud_scaler = StandardScaler().fit(audience_svd[idx_train])
aud_train, aud_test = aud_scaler.transform(audience_svd[idx_train]), aud_scaler.transform(audience_svd[idx_test])

# type + episodes + members, scaled (fit on training anime only)
meta = np.hstack([type_onehot, episodes, members])
meta_scaler = StandardScaler().fit(meta[idx_train])
meta_train, meta_test = meta_scaler.transform(meta[idx_train]), meta_scaler.transform(meta[idx_test])

feature_sets = {
    "Text (name TF-IDF)": (text_train, text_test),
    "Audience (SVD)":     (csr_matrix(aud_train), csr_matrix(aud_test)),
    "All features":       (hstack([meta_train, text_train, aud_train]).tocsr(),
                           hstack([meta_test,  text_test,  aud_test]).tocsr()),
}
for fs_name, (Xtr, Xte) in feature_sets.items():
    print(f"{fs_name:20s} train {Xtr.shape}  test {Xte.shape}")

# 1. Define the three models
base_clf = LogisticRegression(max_iter=1000, solver='lbfgs', random_state=userseed)

# 2. Train, predict, and evaluate on each feature set
results = []
model_preds = {}

for fs_name, (X_train, X_test) in feature_sets.items():
    models = {
        "Always-Most-Common Baseline": OneVsRestClassifier(
            DummyClassifier(strategy="prior")
        ),
        "Binary Relevance": OneVsRestClassifier(
            base_clf, n_jobs=-1
        ),
        "Classifier Chain": ClassifierChain(
            base_clf, order="random", random_state=userseed
        )
    }

    for name, model in models.items():
        model.fit(X_train, Y_train)
        Y_pred = model.predict(X_test)
        model_preds[(fs_name, name)] = Y_pred

        results.append({
            "Features": fs_name,
            "Model": name,
            "Micro F1": f1_score(Y_test, Y_pred, average="micro", zero_division=0),
            "Macro F1": f1_score(Y_test, Y_pred, average="macro", zero_division=0),
            "Subset Accuracy": accuracy_score(Y_test, Y_pred),
            "Hamming Loss": hamming_loss(Y_test, Y_pred)
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

best = ("All features", "Classifier Chain")
report = classification_report(
    Y_test, model_preds[best],
    target_names=mlb.classes_, output_dict=True, zero_division=0
)

per_class_f1 = {genre: m["f1-score"] for genre, m in report.items() if genre in mlb.classes_}

print(f"Per-genre F1 for {best[1]} on {best[0]}")
for genre, f1 in sorted(per_class_f1.items(), key=lambda kv: kv[1], reverse=True):
    print(f"  {genre:15s} {f1:.3f}")

### 📝 Analysis Explanation (1b)

**Instructions:** Why is plain accuracy a poor metric here, and which metric would have flattered your baseline? Which feature set carries more genre signal, and why? Which genres does your best model predict well, and which does it struggle with? Did the classifier chain help?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 2: Transfer Learning & Semi-Supervised Learning  (2 points)

### Problem 2a: Transfer Learning with a Learning Curve

**Instructions:**

Compare three strategies on CIFAR-10 for several amounts of labeled data: (i) a small CNN or ResNet18 trained **from scratch**, (ii) a **pretrained ResNet18 with a frozen encoder** and a new classification layer (feature extraction), and (iii) the pretrained ResNet18 **fine-tuned** with a small learning rate. Use 100, 500, and 2,000 labeled training images (class-balanced), evaluate on the same 2,000 held-out test images, and repeat each setting over 3 seeds.

In [ ]:
# TODO: Load CIFAR-10; build balanced labeled subsets of 100, 500, and 2000 images; fix a 2000-image test set
# TODO: Train from scratch, frozen-encoder + new head, and fine-tuned ResNet18 for each size and seed (a few epochs each)
# TODO: Record test accuracy; compute mean and standard deviation over the 3 seeds
# TODO: Plot accuracy against number of labeled images, one line per strategy, with error bars
# YOUR CODE HERE:
transform = T.Compose([
    T.Resize(224),                       # ResNet18 was pretrained on 224x224 images
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406],   # ImageNet mean/std, needed for the pretrained ResNet18
                [0.229, 0.224, 0.225]),
])

# torchvision's CIFAR-10 loader triggers a NumPy 2.4 deprecation warning inside the library (not from this code)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=np.exceptions.VisibleDeprecationWarning)
    train_full = torchvision.datasets.CIFAR10(root="data", train=True,  download=True, transform=transform)
    test_full  = torchvision.datasets.CIFAR10(root="data", train=False, download=True, transform=transform)

# --- 1. Balanced train subsets (100, 500, 2000), a new draw for each seed ---
train_targets = np.array(train_full.targets)
sizes = [100, 500, 2000]
seeds = [userseed, userseed + 1, userseed + 2]

def make_train_subsets(seed):
    rng = np.random.default_rng(seed)
    train_per_class = [rng.permutation(np.where(train_targets == c)[0]) for c in range(10)]
    subsets = {}
    for size in sizes:
        idx = np.concatenate([c[:size // 10] for c in train_per_class])   # size/10 per class
        rng.shuffle(idx)
        subsets[size] = Subset(train_full, idx)
    return subsets

train_subsets = {}                     # keyed by (size, seed)
for seed in seeds:
    for size, subset in make_train_subsets(seed).items():
        train_subsets[(size, seed)] = subset

# --- 2. Fixed balanced 2000-image test set (built once, same for every run) ---
test_rng = np.random.default_rng(userseed)
test_targets = np.array(test_full.targets)
test_per_class = [test_rng.permutation(np.where(test_targets == c)[0]) for c in range(10)]

test_idx = np.concatenate([c[:200] for c in test_per_class])    # 200 per class
test_rng.shuffle(test_idx)

test_2000 = Subset(test_full, test_idx)

# Quick check
for (size, seed), subset in train_subsets.items():
    print(f"size {size:4d}  seed {seeds.index(seed) + 1}  n = {len(subset)}")
print("test:", len(test_2000))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Fixed test loader (2,000 balanced test images)
test_loader = DataLoader(test_2000, batch_size=128, shuffle=False, num_workers=0, pin_memory=True)

regimes = ["scratch", "frozen_head", "finetune"]
epochs = 5  # "a few epochs each"

results_2a = []

for seed in seeds:                      # userseed, userseed + 1, userseed + 2 (from TODO 1)
    seed_num = seeds.index(seed) + 1    # print 1/2/3 instead of the seed value

    for size in sizes:
        # Balanced subset for this size and seed (built in TODO 1)
        train_loader = DataLoader(
            train_subsets[(size, seed)],
            batch_size=32,
            shuffle=True,
            num_workers=0,              # worker processes often hang in Jupyter on Windows
            pin_memory=True
        )

        for regime in regimes:
            # Reset seed before each model init so initialization/shuffling is fair
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)

            # 1. Initialize ResNet18 according to regime
            if regime == "scratch":
                model = resnet18(weights=None)
                model.fc = nn.Linear(model.fc.in_features, 10)
                lr = 1e-3

            elif regime == "frozen_head":
                model = resnet18(weights=ResNet18_Weights.DEFAULT)
                for param in model.parameters():
                    param.requires_grad = False
                model.fc = nn.Linear(model.fc.in_features, 10)  # New head has requires_grad=True
                lr = 1e-3

            elif regime == "finetune":
                model = resnet18(weights=ResNet18_Weights.DEFAULT)
                model.fc = nn.Linear(model.fc.in_features, 10)
                lr = 1e-4  # Lower LR so pretrained backbone weights aren't destroyed

            model = model.to(device)

            # Optimize only trainable parameters
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=lr,
                weight_decay=1e-4
            )
            criterion = nn.CrossEntropyLoss()

            # 2. Train for a few epochs
            for epoch in range(epochs):
                if regime == "frozen_head":
                    model.eval()        # keep the frozen backbone's BatchNorm statistics fixed
                    model.fc.train()    # only the new head trains
                else:
                    model.train()

                for x, y in train_loader:
                    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    optimizer.zero_grad()
                    loss = criterion(model(x), y)
                    loss.backward()
                    optimizer.step()

            # 3. Evaluate on fixed 2,000-image test set
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for x, y in test_loader:
                    x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                    preds = model(x).argmax(dim=1)
                    correct += (preds == y).sum().item()
                    total += y.size(0)

            acc = correct / total
            results_2a.append({"regime": regime, "size": size, "seed": seed_num, "test_acc": acc})
            print(f"Seed {seed_num} | Size {size:4d} | {regime:11s} | Test Acc: {acc:.4f}")

# Summarize mean ± std across seeds
df_2a = pd.DataFrame(results_2a)
summary = df_2a.groupby(["regime", "size"])["test_acc"].agg(["mean", "std"]).reset_index()
print("\n=== Summary (Mean ± Std across seeds) ===")
print(summary.to_string(index=False))


# Ensure summary is computed and sorted by subset size
summary = (
    df_2a.groupby(["regime", "size"])["test_acc"]
    .agg(["mean", "std"])
    .reset_index()
    .sort_values("size")
)

labels_map = {
    "scratch": "From Scratch",
    "frozen_head": "Frozen Encoder + New Head",
    "finetune": "Fine-Tuned ResNet-18"
}
markers_map = {
    "scratch": "o",
    "frozen_head": "s",
    "finetune": "^"
}

plt.figure(figsize=(8, 5))

for regime in ["scratch", "frozen_head", "finetune"]:
    sub = summary[summary["regime"] == regime]
    plt.errorbar(
        sub["size"],
        sub["mean"],
        yerr=sub["std"].fillna(0),
        label=labels_map[regime],
        marker=markers_map[regime],
        linewidth=2,
        markersize=7,
        capsize=4
    )

plt.xscale("log")  # Log scale spaces 100, 500, and 2000 evenly
plt.xticks([100, 500, 2000], ["100", "500", "2,000"])
plt.xlabel("Number of Labeled Training Images")
plt.ylabel("Test Accuracy (on Fixed 2,000-Image Test Set)")
plt.title("CIFAR-10 Test Accuracy vs. Labeled Subset Size")
plt.ylim(0, 1.0)
plt.grid(True, linestyle="--", alpha=0.5)
plt.legend()
plt.tight_layout()
plt.savefig("resnet18_learning_curves.png", dpi=300)
plt.show()

### 📝 Analysis Explanation (2a)

**Instructions:** At which label budget does transfer help most, and when does the gap narrow? When would you choose feature extraction over full fine-tuning? Why is repeating over seeds necessary here?

**Your Analysis:** [Write your interpretation here.]

### Problem 2b: Semi-Supervised Learning

**Instructions:**

Using the Breast Cancer Wisconsin dataset, hide 80% of the training labels (treat them as unlabeled, label -1). Apply self-training: fit on the labeled 20%, add high-confidence pseudo-labels, and repeat. Compare (i) a model trained only on the 20% labeled data, (ii) the self-trained model at two confidence thresholds (for example 0.7 and 0.95), and (iii) an oracle trained on all labels. Repeat over 5 random splits and report mean and standard deviation.

In [ ]:
# TODO: Load load_breast_cancer(), split into train/test, hide 80% of train labels (-1)
# TODO: Fit the labeled-only baseline, SelfTrainingClassifier at two thresholds, and the all-labels oracle
# TODO: Repeat over 5 random splits; report mean and std of test accuracy in one table
# TODO: Report how many pseudo-labels were added and what share of them were correct (you know the true labels)
# YOUR CODE HERE:

# Load the Breast Cancer Wisconsin dataset
X_bc, y_bc = load_breast_cancer(return_X_y=True)
print("Shape:", X_bc.shape, "| Class counts:", np.bincount(y_bc))

#  Stratified train/test split (e.g., 80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=userseed
)

# Hide 80% of training labels by setting them to -1 (stratified so both classes remain represented)
_, unlabeled_idx = train_test_split(
    np.arange(len(y_train)),
    test_size=0.80,
    stratify=y_train,
    random_state=userseed
)

y_train_semi = y_train.copy()
y_train_semi[unlabeled_idx] = -1

# Quick verification
print("Train shape:", X_train.shape, "| Test shape:", X_test.shape)
print("Labeled count:", np.sum(y_train_semi != -1), "| Unlabeled (-1) count:", np.sum(y_train_semi == -1))
print("Class distribution in remaining 20% labeled:", np.bincount(y_train_semi[y_train_semi != -1]))

# Scale features (fit on X_train only to avoid leakage)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Mask for the 20% labeled training samples
labeled_mask = (y_train_semi != -1)

#  Define the four models
models = {
    "Labeled-Only Baseline (20%)": LogisticRegression(max_iter=1000, random_state=userseed),
    "Self-Training (threshold=0.75)": SelfTrainingClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=userseed),
        threshold=0.75
    ),
    "Self-Training (threshold=0.95)": SelfTrainingClassifier(
        estimator=LogisticRegression(max_iter=1000, random_state=userseed),
        threshold=0.95
    ),
    "All-Labels Oracle (100%)": LogisticRegression(max_iter=1000, random_state=userseed)
}

#  Fit each model on its respective labels and evaluate on X_test
results_2b = []

for name, model in models.items():
    if "Labeled-Only" in name:
        model.fit(X_train_scaled[labeled_mask], y_train_semi[labeled_mask])
    elif "Self-Training" in name:
        model.fit(X_train_scaled, y_train_semi)  # Uses both labeled and -1 unlabeled rows
    else:  # Oracle
        model.fit(X_train_scaled, y_train)

    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    # Track how many unlabeled points were pseudo-labeled (for SelfTrainingClassifier)
    n_pseudo = (
        int((model.transduction_[~labeled_mask] != -1).sum())
        if hasattr(model, "transduction_")
        else "-"
    )

    results_2b.append({
        "Model": name,
        "Pseudo-Labeled": n_pseudo,
        "Test Accuracy": accuracy_score(y_test, y_pred),
        "Test F1": f1_score(y_test, y_pred),
        "Test ROC-AUC": roc_auc_score(y_test, y_prob)
    })

results_2b_df = pd.DataFrame(results_2b)
print(results_2b_df.to_string(index=False))

# ---------- TODO 3 + 4: repeat over 5 random splits, track pseudo-labels ----------
seeds_2b = [userseed + i for i in range(5)]
records_2b = []

for seed in seeds_2b:
    # 1. Train/test split for this seed
    X_train, X_test, y_train, y_test = train_test_split(
        X_bc, y_bc, test_size=0.2, stratify=y_bc, random_state=seed
    )

    # 2. Hide 80% of training labels (-1)
    _, unlabeled_idx = train_test_split(
        np.arange(len(y_train)),
        test_size=0.80,
        stratify=y_train,
        random_state=seed
    )
    y_train_semi = y_train.copy()
    y_train_semi[unlabeled_idx] = -1
    labeled_mask = (y_train_semi != -1)

    # Scale features (fit on X_train only)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Fit all four models
    models = {
        "Labeled-Only Baseline (20%)": LogisticRegression(max_iter=1000, random_state=seed),
        "Self-Training (threshold=0.75)": SelfTrainingClassifier(
            estimator=LogisticRegression(max_iter=1000, random_state=seed),
            threshold=0.75
        ),
        "Self-Training (threshold=0.95)": SelfTrainingClassifier(
            estimator=LogisticRegression(max_iter=1000, random_state=seed),
            threshold=0.95
        ),
        "All-Labels Oracle (100%)": LogisticRegression(max_iter=1000, random_state=seed)
    }

    for name, model in models.items():
        if "Labeled-Only" in name:
            model.fit(X_train_scaled[labeled_mask], y_train_semi[labeled_mask])
        elif "Self-Training" in name:
            model.fit(X_train_scaled, y_train_semi)
        else:
            model.fit(X_train_scaled, y_train)

        acc = accuracy_score(y_test, model.predict(X_test_scaled))

        # TODO 4: pseudo-labels = originally unlabeled points that self-training gave a label
        n_pseudo, share_correct = np.nan, np.nan
        if "Self-Training" in name:
            pseudo_mask = (~labeled_mask) & (model.transduction_ != -1)
            n_pseudo = int(pseudo_mask.sum())
            share_correct = (model.transduction_[pseudo_mask] == y_train[pseudo_mask]).mean()

        records_2b.append({"Seed": seeds_2b.index(seed) + 1, "Model": name, "Test Accuracy": acc,
                           "Pseudo-Labels Added": n_pseudo, "Share Correct": share_correct})

# Aggregate mean and std across the 5 random splits in a single table
results_2b_df = pd.DataFrame(records_2b)
summary_2b = (
    results_2b_df.groupby("Model", sort=False)["Test Accuracy"]
    .agg(Mean="mean", Std="std")
    .reset_index()
)
summary_2b["Mean ± Std"] = summary_2b.apply(
    lambda r: f"{r['Mean']:.4f} ± {r['Std']:.4f}", axis=1
)
print("=== Test accuracy over 5 random splits ===")
print(summary_2b.to_string(index=False))

# pseudo-label counts and correctness (self-training models only)
pseudo_summary = (
    results_2b_df[results_2b_df["Model"].str.contains("Self-Training")]
    .groupby("Model", sort=False)[["Pseudo-Labels Added", "Share Correct"]]
    .agg(["mean", "std"])
)
print("\n=== Pseudo-labels (out of 364 unlabeled points per split) ===")
print(pseudo_summary.round(3).to_string())

### 📝 Analysis Explanation (2b)

**Instructions:** Did self-training beat the labeled-only baseline, and by how much compared with the spread across splits? Use the share of correct pseudo-labels to explain the result. Under what conditions would self-training hurt?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 3: Graph & Network Mining  (2 points)

### Problem 3: Graph & Network Mining

**Instructions:**

Load Zachary's Karate Club graph. Compute degree centrality, betweenness centrality, and PageRank; detect communities with Louvain (`networkx.community.louvain_communities`) and compare them with the known club split stored in the node attribute `club` (use normalized mutual information) and report the modularity of the Louvain partition. Compare the top-5 nodes by PageRank with the top-5 by degree and by betweenness. Then run a small **link-prediction** test: hide 20% of the edges, score the hidden edges and random non-edges with common neighbors and the Adamic-Adar index, and report ROC AUC for both. Visualize the graph colored by community.

In [ ]:
# TODO: Load nx.karate_club_graph(); compute degree, betweenness, and PageRank for every node
# TODO: Run Louvain; compute NMI against the node attribute 'club'; plot the graph colored by community
# TODO: Hide 20% of the edges, score with the Adamic-Adar index, and compute ROC AUC against sampled non-edges (repeat over several random hides)
# YOUR CODE HERE:

# Load Zachary's Karate Club graph
G = nx.karate_club_graph()
print("Nodes:", G.number_of_nodes(), "| Edges:", G.number_of_edges())
print("Club sizes:", pd.Series(nx.get_node_attributes(G, "club")).value_counts().to_dict())

# Compute nodes
degrees = dict(G.degree())
deg_norm = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G)
pr = nx.pagerank(G, weight=None)   # unweighted, same as degree and betweenness
club = nx.get_node_attributes(G, "club")

# Combine into a single DataFrame indexed by node
nodes_df = pd.DataFrame({
    "node": list(G.nodes()),
    "club": [club[n] for n in G.nodes()],
    "degree": [degrees[n] for n in G.nodes()],
    "degree_centrality": [deg_norm[n] for n in G.nodes()],
    "betweenness": [btw[n] for n in G.nodes()],
    "pagerank": [pr[n] for n in G.nodes()]
})

# Compare the top-5 nodes under each measure
top5 = pd.DataFrame({
    "Rank": range(1, 6),
    "Degree": nodes_df.nlargest(5, "degree_centrality")["node"].values,
    "Betweenness": nodes_df.nlargest(5, "betweenness")["node"].values,
    "PageRank": nodes_df.nlargest(5, "pagerank")["node"].values,
})
print("\nTop-5 nodes by each centrality measure:")
print(top5.to_string(index=False))

# Run Louvain community detection
communities = nx.community.louvain_communities(G, seed=userseed)   # uses edge weights (default)
print(f"Detected {len(communities)} communities:", [sorted(c) for c in communities])

# Map each node to its Louvain community ID (0, 1, ...)
node_to_comm = {}
for comm_id, nodes in enumerate(communities):
    for n in nodes:
        node_to_comm[n] = comm_id

# Compute NMI against the ground-truth 'club' attribute
nodes_ordered = list(G.nodes())
true_clubs = [G.nodes[n]["club"] for n in nodes_ordered]
pred_comms = [node_to_comm[n] for n in nodes_ordered]

nmi = normalized_mutual_info_score(true_clubs, pred_comms)
print(f"NMI vs. ground-truth 'club': {nmi:.4f}")

# Modularity of the Louvain partition (same edge weights Louvain used)
modularity = nx.community.modularity(G, communities)
print(f"Modularity of Louvain partition: {modularity:.4f}")

# Plot the graph colored by detected Louvain community
plt.figure(figsize=(9, 6))
pos = nx.spring_layout(G, seed=userseed)

# Draw edges and nodes colored by community ID
nx.draw_networkx_edges(G, pos, alpha=0.35, edge_color="gray")
nodes_plot = nx.draw_networkx_nodes(
    G,
    pos,
    nodelist=nodes_ordered,
    node_color=pred_comms,
    cmap=plt.cm.Set2,
    node_size=500,
    edgecolors="black"
)
nx.draw_networkx_labels(G, pos, font_size=9, font_weight="bold")

# Add legend mapping community colors
handles = [
    plt.Line2D([0], [0], marker="o", color="w", label=f"Community {i}",
               markerfacecolor=plt.cm.Set2(i / max(1, len(communities) - 1)),
               markeredgecolor="black", markersize=10)
    for i in range(len(communities))
]
plt.legend(handles=handles, title="Louvain Communities", loc="best")
plt.title(f"Zachary's Karate Club — Louvain Communities (NMI = {nmi:.4f})")
plt.axis("off")
plt.tight_layout()
plt.show()


# Prepare canonical edge list (u < v) and all non-edges (u < v)
all_edges = [tuple(sorted(e)) for e in G.edges()]
all_non_edges = [tuple(sorted(e)) for e in nx.non_edges(G)]

n_hide = int(0.20 * len(all_edges))  # 20% of 78 edges = 15 hidden edges
seeds_lp = [userseed + i for i in range(10)]
aucs_cn, aucs_aa = [], []

for hide_num, seed in enumerate(seeds_lp, start=1):
    rng = np.random.default_rng(seed)

    # 1. Randomly select 20% of existing edges to hide
    hide_idx = rng.choice(len(all_edges), size=n_hide, replace=False)
    hidden_edges = [all_edges[i] for i in hide_idx]

    # Build residual training graph with hidden edges removed
    G_train = G.copy()
    G_train.remove_edges_from(hidden_edges)

    # 2. Sample an equal number of non-edges as negative examples
    neg_idx = rng.choice(len(all_non_edges), size=n_hide, replace=False)
    sampled_non_edges = [all_non_edges[i] for i in neg_idx]

    # 3. Score hidden positive edges and sampled negative non-edges on the training graph
    eval_pairs = hidden_edges + sampled_non_edges
    y_true = [1] * len(hidden_edges) + [0] * len(sampled_non_edges)

    # Common neighbors: number of shared neighbors in the training graph
    cn_scores = [len(list(nx.common_neighbors(G_train, u, v))) for u, v in eval_pairs]
    # Adamic-Adar: shared neighbors weighted by 1 / log(degree); returns (u, v, score) tuples
    aa_scores = [score for _, _, score in nx.adamic_adar_index(G_train, eval_pairs)]

    # 4. Compute ROC AUC for this random hide
    auc_cn = roc_auc_score(y_true, cn_scores)
    auc_aa = roc_auc_score(y_true, aa_scores)
    aucs_cn.append(auc_cn)
    aucs_aa.append(auc_aa)
    print(f"Hide {hide_num:2d} | Common Neighbors AUC: {auc_cn:.4f} | Adamic-Adar AUC: {auc_aa:.4f}")

print(f"\nMean ROC AUC over {len(seeds_lp)} hides:")
print(f"  Common Neighbors: {np.mean(aucs_cn):.4f} ± {np.std(aucs_cn, ddof=1):.4f}")
print(f"  Adamic-Adar:      {np.mean(aucs_aa):.4f} ± {np.std(aucs_aa, ddof=1):.4f}")

### 📝 Analysis Explanation (3)

**Instructions:** Which nodes are most central by each measure, and why might the three measures disagree? What do the detected communities represent, and how closely do they match the real split? Is the link-prediction AUC trustworthy on a graph this small?

**Your Analysis:** [Write your interpretation here.]

---
## Problem 4: Federated Learning  (2 points)

### Problem 4: Federated Learning

**Instructions:**

Use the Forest Covertype data (take a random 60,000-row subsample; scale the features). Define **four clients by wilderness area** (the four one-hot columns 10 to 13), so each client has a different class mix. Train a logistic regression locally on each client, then average the coefficients and intercepts (weighted by client size) into one federated model. Compare test accuracy of (i) the federated average, (ii) a model trained on the pooled data, and (iii) the average accuracy of the individual local models, each evaluated on a test set drawn from all four areas. Then repeat the comparison with an **IID split** (four random clients of equal size) as a control.

In [ ]:
# TODO: Fetch Covertype; take a random 60,000-row subsample; scale features; hold out a test set from all areas
# TODO: Build four clients by wilderness area; train one logistic regression per client
# TODO: Average coefficients and intercepts weighted by client size; evaluate the federated model
# TODO: Train the pooled model; evaluate it and the individual local models
# TODO: Repeat with four random (IID) clients and report both settings in one table
# YOUR CODE HERE:

# Fetch the Forest Covertype dataset (581,012 rows, 54 features, 7 forest types)
covtype = fetch_covtype()
X_cov, y_cov = covtype.data, covtype.target

# Random 60,000-row subsample (seeded)
cov_rng = np.random.default_rng(userseed)
sub_idx = cov_rng.choice(len(X_cov), size=60000, replace=False)
X_cov, y_cov = X_cov[sub_idx], y_cov[sub_idx]

# Wilderness area of each row (one-hot columns 10 to 13 -> area 0..3), taken before scaling
area_cov = X_cov[:, 10:14].argmax(axis=1)

# Hold out a 20% test set drawn from all four areas (stratified by area)
Xc_train, Xc_test, yc_train, yc_test, area_train, area_test = train_test_split(
    X_cov, y_cov, area_cov, test_size=0.2, stratify=area_cov, random_state=userseed
)

# Scale features (fit on the training rows only)
cov_scaler = StandardScaler().fit(Xc_train)
Xc_train = cov_scaler.transform(Xc_train)
Xc_test = cov_scaler.transform(Xc_test)

# Quick check
print("Train:", Xc_train.shape, "| Test:", Xc_test.shape)
print("Rows per wilderness area (train):", np.bincount(area_train).tolist())
print("Rows per wilderness area (test): ", np.bincount(area_test).tolist())
print("Forest types in test:", sorted(np.unique(yc_test).tolist()))

all_classes = np.array([1, 2, 3, 4, 5, 6, 7])
n_classes = len(all_classes)
n_features = Xc_train.shape[1]

# Define client partitions for both settings:
#    - Non-IID: 4 clients grouped by wilderness area (area_train == 0, 1, 2, 3)
#    - IID:     4 clients formed by randomly splitting Xc_train into 4 equal shards
iid_rng = np.random.default_rng(userseed)
perm = iid_rng.permutation(len(Xc_train))
iid_splits = np.array_split(perm, 4)

partitions = {
    "Non-IID (Wilderness Area)": [np.where(area_train == a)[0] for a in range(4)],
    "IID (Random Split)": iid_splits
}

# Train the centralized Pooled Model once on all training rows
pooled_clf = LogisticRegression(max_iter=1000, random_state=userseed)
pooled_clf.fit(Xc_train, yc_train)
pooled_pred = pooled_clf.predict(Xc_test)
pooled_acc = accuracy_score(yc_test, pooled_pred)
pooled_f1 = f1_score(yc_test, pooled_pred, average="macro")

# Train local models, perform size-weighted FedAvg, and collect metrics
records_fl = []

for setting, client_indices_list in partitions.items():
    total_samples = sum(len(idx) for idx in client_indices_list)
    fed_coef = np.zeros((n_classes, n_features), dtype=np.float64)
    fed_intercept = np.zeros(n_classes, dtype=np.float64)
    local_accs, local_f1s = [], []

    for client_id, idx in enumerate(client_indices_list):
        X_k, y_k = Xc_train[idx], yc_train[idx]
        weight = len(idx) / total_samples

        # Train local logistic regression
        local_clf = LogisticRegression(max_iter=1000, random_state=userseed)
        local_clf.fit(X_k, y_k)

        # Evaluate local model on the global test set
        local_pred = local_clf.predict(Xc_test)
        local_accs.append(accuracy_score(yc_test, local_pred))
        local_f1s.append(f1_score(yc_test, local_pred, average="macro", zero_division=0))
        records_fl.append({
            "Setting": setting,
            "Model": f"Client {client_id} (n={len(idx)})",
            "Test Accuracy": accuracy_score(yc_test, local_pred),
            "Macro F1": f1_score(yc_test, local_pred, average="macro", zero_division=0)
        })

        # Align local coef_ and intercept_ to all 7 classes before weighted averaging
        aligned_coef = np.zeros((n_classes, n_features), dtype=np.float64)
        aligned_intercept = np.full(n_classes, -10.0, dtype=np.float64)  # Low prior for unseen classes

        for local_row, cls_label in enumerate(local_clf.classes_):
            global_row = np.where(all_classes == cls_label)[0][0]
            aligned_coef[global_row] = local_clf.coef_[local_row]
            aligned_intercept[global_row] = local_clf.intercept_[local_row]

        fed_coef += weight * aligned_coef
        fed_intercept += weight * aligned_intercept

    # (iii) Average accuracy of the individual local models
    records_fl.append({
        "Setting": setting,
        "Model": "Average of Local Models",
        "Test Accuracy": np.mean(local_accs),
        "Macro F1": np.mean(local_f1s)
    })

    # Build the aggregated Federated Model
    fed_clf = LogisticRegression()
    fed_clf.classes_ = all_classes
    fed_clf.coef_ = fed_coef
    fed_clf.intercept_ = fed_intercept

    fed_pred = fed_clf.predict(Xc_test)
    records_fl.append({
        "Setting": setting,
        "Model": "Federated Average (FedAvg)",
        "Test Accuracy": accuracy_score(yc_test, fed_pred),
        "Macro F1": f1_score(yc_test, fed_pred, average="macro", zero_division=0)
    })

    # Add Pooled reference row per setting for side-by-side comparison
    records_fl.append({
        "Setting": setting,
        "Model": "Pooled Oracle (Centralized)",
        "Test Accuracy": pooled_acc,
        "Macro F1": pooled_f1
    })

summary_fl = pd.DataFrame(records_fl)
print(summary_fl.to_string(index=False))


### 📝 Analysis Explanation (4)

**Instructions:** How much accuracy did federating cost in the non-IID split compared with the IID split? Why does non-IID data hurt a single round of averaging? Federated learning trades some accuracy for what practical benefit, and in what real-world scenario would you accept that trade?

**Your Analysis:** [Write your interpretation here.]

---
## Results Interpretation  (2 points)

**Instructions:** Step back from the individual sub-parts and write a short, integrated interpretation of this assignment's results. Address: (1) which of the paradigms you explored seems most directly applicable to a project you might actually pursue; (2) one real constraint (data, compute, or privacy) that would push you toward one of these paradigms over standard supervised learning; and (3) which paradigm you found hardest to get working well, and your best guess as to why.

Write 150-250 words. This section is graded on the quality of your reasoning and how well you tie your specific numbers and figures back into a clear conclusion, not on reaching any particular "correct" answer.

**Your Interpretation:** [Write your response here.]

---
## AI Usage Statement (required)

Briefly state which AI tools you used (if any) and for what, consistent with the course AI Tool Policy.

> [Example: "GitHub Copilot was used for boilerplate plotting code. All analysis and interpretation are my own work."]